# Braket: 10-qubit VQC inference + XEB (from existing OpenQASM 3 files)

Sibling of `braket-benchmarks.ipynb`. **The only difference**: instead of
regenerating the VQC and XEB circuits in pure Qiskit from the same seeds,
this notebook loads the exact `.qasm` files in
`data/circuits/` (the same artefacts that were submitted to
IBM Quantum) via `qiskit.qasm3.load`, transpiles them for the chosen Braket
backend, and submits them through `BackendSamplerV2`.

Use this notebook when you want **byte-identical circuits between IBM and
Braket runs** — e.g. for cross-device comparison plots where regeneration
drift would be a confounder.

**What it does**

1. **Setup** — installs Python deps once, defines paths and hyper-parameters.
2. **Discover QASM** — globs `qvc_test_*.qasm` and `xeb_d*_c*.qasm` under
   `QASM_DIR` (default: the `data/circuits/` folder next to this
   notebook).  Reads:
   - VQC `true_label` from the file header comment `// ... (true label X).`
   - XEB depth `d` from the filename tag `xeb_dDD_cCCC.qasm`.
3. **Per device** — for each Braket backend you list:
   - **VQC** — loads each `qvc_test_NNN.qasm`, transpiles for the Braket
     target, submits with shots, saves counts/predictions/summary under
     `braket_runs/<device_slug>/vqc/`.
   - **XEB** — loads each `xeb_dDD_cCCC.qasm`, strips final measurements to
     compute ideal probabilities via `Statevector`, re-adds measurements,
     transpiles, submits, and computes the linear-XEB fidelity per circuit.
     Statistics are aggregated per depth under `.../xeb/`.

**Important — no weights file is needed.**  The VQC `.qasm` files are
fully bound (image angles + trained weights already baked in), so this
notebook does **not** load `weights_10q.npy` or do any training.

**Credentials**

On Braket notebooks the execution role already has `braket:CreateQuantumTask`.
Locally run `aws configure` with keys that can call Braket in your chosen region.


In [ ]:
# Run once per fresh kernel (Braket notebook images may already have some packages).
# qiskit-qasm3-import is required for qiskit.qasm3.load() of files that use
# custom gate definitions like the `r(p0, p1)` block in the XEB QASM.
%pip install -q "qiskit>=1.2" qiskit-qasm3-import qiskit-braket-provider amazon-braket-sdk scipy matplotlib


In [ ]:
from __future__ import annotations

import csv
import json
import re
import time
from datetime import datetime, timezone
from pathlib import Path
from typing import Any, Dict, List, Optional, Tuple

import numpy as np
import matplotlib.pyplot as plt
from scipy.optimize import curve_fit

from qiskit import QuantumCircuit, qasm3
from qiskit.primitives import BackendSamplerV2
from qiskit.quantum_info import Statevector
from qiskit.transpiler.passes import RemoveBarriers
from qiskit.transpiler.preset_passmanagers import generate_preset_pass_manager

# ---------------------------------------------------------------------------
# Root folder for all Braket benchmark outputs (timestamped subfolder).
# ---------------------------------------------------------------------------
RUN_STAMP = datetime.now(timezone.utc).strftime("%Y%m%d_%H%M%SZ")
OUTPUT_ROOT = Path("braket_runs_from_qasm") / RUN_STAMP
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
print("Results will be written under:", OUTPUT_ROOT.resolve())


## Configuration — edit this cell only


In [ ]:
# ---------------------------------------------------------------------------
# Braket backends to benchmark (Qiskit-Braket short names or device ARNs).
# Examples: "SV1", "DM1", "TN1", "local" (BraketLocalBackend), or an ARN string.
# ---------------------------------------------------------------------------
BRAKET_DEVICES: List[str] = [
    "SV1",          # cloud simulator — good first smoke test
    # "DM1",
    # "arn:aws:braket:eu-north-1::device/qpu/iqm/Garnet",
    # "arn:aws:braket:eu-north-1::device/qpu/iqm/Emerald",
    # "arn:aws:braket:us-west-1::device/qpu/rigetti/Ankaa-3",
]

# If get_backend says "No backend matches", set the region where Braket lists
# devices (SV1/DM1/TN1 in us-east-1; IQM in eu-north-1; Rigetti in us-west-1).
BRAKET_REGION: Optional[str] = None   # e.g. "us-east-1"

# Shots
SHOTS_VQC = 4000
SHOTS_XEB = 4000

# How many VQC test images to run on each device (None = all qvc_test_*.qasm).
VQC_TEST_LIMIT: Optional[int] = None

# How many XEB circuits per depth to run (None = all available for that depth).
XEB_PER_DEPTH_LIMIT: Optional[int] = None

# Batch size for SamplerV2 (circuits per quantum task batch).
CHUNK_SIZE = 25

# ---------------------------------------------------------------------------
# Where to find the OpenQASM 3 files.
#
# Default looks for `data/circuits/` next to this notebook *and* under
# `data/circuits/` (so this works regardless of whether you
# upload only the notebook or the entire repo to the Braket instance).
# Set QASM_DIR explicitly to override.
# ---------------------------------------------------------------------------
QASM_DIR: Optional[Path] = None   # e.g. Path("/home/ec2-user/SageMaker/things_to_run_on_ibm")

# Optional S3 prefix (s3://bucket/prefix/) to download the QASM files from at
# runtime — useful if you cannot ship a folder alongside the notebook.  All
# *.qasm objects under the prefix are mirrored to /tmp/data/circuits/.
QASM_S3_PREFIX: Optional[str] = None

# Transpiler optimisation level for each Braket backend target.
TRANSPILE_OPTLEVEL = 1

# Number of qubits expected in every loaded circuit (sanity check).
NUM_QUBITS = 10

# Optional: try to import Braket cost tracking
try:
    from braket.tracking import Tracker
    _TRACKER = Tracker().start()
except Exception:
    _TRACKER = None


## Optional: discover backend names (run if `get_backend` fails)

`BraketProvider().get_backend(...)` resolves devices through **`AwsDevice.get_devices`**, which is **region-sensitive** and uses **exact names** (e.g. some builds use `dm1` not `DM1`). If the list below is empty, set `BRAKET_REGION = "us-east-1"` in the config cell and re-run this cell.


In [ ]:
import os

_reg = globals().get("BRAKET_REGION")
if _reg:
    os.environ["AWS_DEFAULT_REGION"] = str(_reg)

print("AWS_DEFAULT_REGION =", os.environ.get("AWS_DEFAULT_REGION", "(not set)"))

from qiskit_braket_provider import BraketProvider

names = sorted({b.name for b in BraketProvider().backends()})
print(len(names), "gate-model backend(s). Copy one into BRAKET_DEVICES:")
for n in names:
    print(" ", repr(n))


## Core helpers — QASM loading + XEB / parity utilities


In [ ]:
# ---------------------------------------------------------------------------
# Discover the directory that holds the OpenQASM files.
# ---------------------------------------------------------------------------
def _resolve_qasm_dir() -> Path:
    # Honour explicit override first.
    explicit = globals().get("QASM_DIR")
    if explicit is not None:
        p = Path(explicit).expanduser().resolve()
        if not p.is_dir():
            raise FileNotFoundError(f"QASM_DIR does not exist: {p}")
        return p

    # Optional: pull from S3.
    s3_prefix = (globals().get("QASM_S3_PREFIX") or "").strip()
    if s3_prefix.startswith("s3://"):
        import boto3
        rest = s3_prefix[5:].rstrip("/")
        bucket, _, key_prefix = rest.partition("/")
        if not key_prefix:
            raise ValueError(f"Bad QASM_S3_PREFIX (need s3://bucket/prefix): {s3_prefix}")
        dest = Path("/tmp/things_to_run_on_ibm")
        dest.mkdir(parents=True, exist_ok=True)
        s3 = boto3.client("s3")
        paginator = s3.get_paginator("list_objects_v2")
        n = 0
        for page in paginator.paginate(Bucket=bucket, Prefix=key_prefix + "/"):
            for obj in page.get("Contents", []) or []:
                key = obj["Key"]
                if not key.endswith(".qasm"):
                    continue
                local = dest / Path(key).name
                s3.download_file(bucket, key, str(local))
                n += 1
        if n == 0:
            raise FileNotFoundError(
                f"No *.qasm objects under s3://{bucket}/{key_prefix}/"
            )
        print(f"Downloaded {n} *.qasm files from {s3_prefix} -> {dest}")
        return dest

    # Auto-discovery: look in plausible locations.
    here = Path.cwd().resolve()
    candidates = [
        here.parent / "data" / "circuits",
        here.parent / "data" / "circuits",
        here.parent / "data" / "circuits",
        here.parent / "data" / "circuits",
    ]
    for c in candidates:
        if c.is_dir():
            return c.resolve()

    raise FileNotFoundError(
        "Could not find data/circuits/. Set QASM_DIR to an absolute path "
        "or QASM_S3_PREFIX to an s3:// URI containing the .qasm files. "
        f"Searched: {[str(c) for c in candidates]}"
    )


# ---------------------------------------------------------------------------
# Filename / header parsers.
# ---------------------------------------------------------------------------
_VQC_FNAME_RE = re.compile(r"qvc_test_(\d+)\.qasm$")
_XEB_FNAME_RE = re.compile(r"xeb_d(\d+)_c(\d+)\.qasm$")
_VQC_LABEL_RE = re.compile(r"true\s+label\s+(-?\d+)", re.IGNORECASE)


def _parse_vqc_index(path: Path) -> int:
    m = _VQC_FNAME_RE.search(path.name)
    if not m:
        raise ValueError(f"Not a VQC test QASM file: {path.name}")
    return int(m.group(1))


def _parse_vqc_label(path: Path) -> int:
    head = path.read_text().splitlines()[:5]
    for line in head:
        m = _VQC_LABEL_RE.search(line)
        if m:
            return int(m.group(1))
    raise ValueError(
        f"Could not parse 'true label' from header of {path.name}. "
        "Expected a comment line like: // ... (true label 1)."
    )


def _parse_xeb_depth_idx(path: Path) -> Tuple[int, int]:
    m = _XEB_FNAME_RE.search(path.name)
    if not m:
        raise ValueError(f"Not an XEB QASM file: {path.name}")
    return int(m.group(1)), int(m.group(2))


# ---------------------------------------------------------------------------
# QASM 3 load helpers.
# ---------------------------------------------------------------------------
# Reusable transpiler pass: every .qasm file in data/circuits/ ends with
# `barrier q[0], ..., q[9];` before the measurements.  qiskit-braket-provider
# forwards Barrier instructions verbatim into the Braket OpenQASM payload, and
# Braket simulators (SV1 / DM1 / TN1) reject tasks containing `barrier` at
# validation time.  Stripping it here keeps the gate sequence byte-identical
# to the IBM submission otherwise.
_REMOVE_BARRIERS = RemoveBarriers()


def load_qasm3_circuit(path: Path) -> QuantumCircuit:
    # qiskit.qasm3.load supports custom gate definitions when qiskit-qasm3-import
    # is installed.  Returns a Qiskit QuantumCircuit with classical register
    # `meas` (10 bits) and final measurements on every qubit.
    qc = qasm3.load(str(path))
    if qc.num_qubits != NUM_QUBITS:
        raise ValueError(
            f"{path.name}: expected {NUM_QUBITS} qubits, got {qc.num_qubits}"
        )
    return _REMOVE_BARRIERS(qc)


def strip_final_measurements(qc: QuantumCircuit) -> QuantumCircuit:
    # Used to compute ideal Born probabilities for XEB.
    return qc.remove_final_measurements(inplace=False)


# ---------------------------------------------------------------------------
# Result post-processing.
# ---------------------------------------------------------------------------
def parity_expectation_from_bitstrings(bits: List[str]) -> float:
    ev = 0.0
    for b in bits:
        s = b.replace(" ", "")
        parity = s.count("1") % 2
        ev += (-1.0) ** parity
    return ev / len(bits)


def xeb_fidelity(p_ideal: np.ndarray, samples: np.ndarray) -> float:
    D = len(p_ideal)
    counts = np.bincount(samples, minlength=D)
    p_meas = counts / counts.sum()
    num = float(np.sum(p_meas * p_ideal) - 1.0 / D)
    den = float(np.sum(p_ideal ** 2) - 1.0 / D)
    return num / den


def ideal_probabilities(circuit_no_meas: QuantumCircuit) -> np.ndarray:
    return np.abs(Statevector.from_instruction(circuit_no_meas).data) ** 2


# ---------------------------------------------------------------------------
# BackendSamplerV2 result access.  Both VQC and XEB QASM files use a classical
# register named `meas`, so `pr.data.meas.get_bitstrings()` works for both.
# Wrapped here so we fall back gracefully if a future qiskit-braket-provider
# release renames the register.
# ---------------------------------------------------------------------------
def _bitstrings_from_pub_result(pr) -> List[str]:
    data = pr.data
    # Fast path: classical register named "meas" (matches QASM declarations).
    if hasattr(data, "meas"):
        return data.meas.get_bitstrings()
    # Fall back: pick the first BitArray-like attribute that exposes
    # get_bitstrings().
    for name in dir(data):
        if name.startswith("_"):
            continue
        attr = getattr(data, name, None)
        if attr is not None and hasattr(attr, "get_bitstrings"):
            return attr.get_bitstrings()
    raise RuntimeError(
        "BackendSamplerV2 result did not expose any get_bitstrings(); "
        "available attributes: "
        + repr([n for n in dir(data) if not n.startswith("_")])
    )


## Braket runners + `run_braket_device`


In [ ]:
def _device_slug(name: str) -> str:
    s = re.sub(r"[^0-9A-Za-z._-]+", "_", name)
    return s[:120] if len(s) > 120 else s


def _get_braket_backend(backend_name: str):
    from qiskit_braket_provider import BraketProvider, BraketLocalBackend
    from qiskit.providers.exceptions import QiskitBackendNotFoundError
    import os

    _reg = globals().get("BRAKET_REGION")
    if _reg:
        os.environ["AWS_DEFAULT_REGION"] = str(_reg)

    if backend_name.lower() == "local":
        return BraketLocalBackend()

    prov = BraketProvider()

    # ARN: try as-is first (Braket / qiskit-braket-provider accept full device ARN).
    if backend_name.startswith("arn:aws:braket:"):
        try:
            return prov.get_backend(backend_name)
        except QiskitBackendNotFoundError:
            pass

    # Short names are case-sensitive in AWS; try a few variants.
    variants: List[str] = []
    for v in (backend_name, backend_name.upper(), backend_name.lower()):
        if v not in variants:
            variants.append(v)
    for cand in variants:
        try:
            return prov.get_backend(cand)
        except QiskitBackendNotFoundError:
            continue

    try:
        all_names = sorted({b.name for b in prov.backends()})
    except Exception as exc:
        raise RuntimeError(
            "Could not list Braket devices. Check IAM (braket:GetDevice, etc.), "
            "AWS credentials, and set BRAKET_REGION (try 'us-east-1'). "
            f"Details: {exc}"
        ) from exc

    raise QiskitBackendNotFoundError(
        f"No backend matching {backend_name!r} (tried {variants}). "
        f"Currently visible gate-model names ({len(all_names)}): {all_names}"
    )


def _sampler_chunks(pubs: List, sampler: BackendSamplerV2) -> List[Any]:
    # Run pubs in chunks; return flat list of pub_results in order.
    out = []
    for i in range(0, len(pubs), CHUNK_SIZE):
        chunk = pubs[i : i + CHUNK_SIZE]
        job = sampler.run(chunk)
        out.extend(job.result())
    return out


# ---------------------------------------------------------------------------
# VQC
# ---------------------------------------------------------------------------
def run_vqc_braket_from_qasm(
    backend_name: str,
    out_vqc: Path,
    qasm_dir: Path,
) -> Dict[str, Any]:
    out_vqc.mkdir(parents=True, exist_ok=True)
    backend = _get_braket_backend(backend_name)
    pm = generate_preset_pass_manager(target=backend.target,
                                      optimization_level=TRANSPILE_OPTLEVEL)
    sampler = BackendSamplerV2(backend=backend,
                               options={"default_shots": SHOTS_VQC})

    files = sorted(qasm_dir.glob("qvc_test_*.qasm"),
                   key=lambda p: _parse_vqc_index(p))
    if not files:
        raise FileNotFoundError(f"No qvc_test_*.qasm files under {qasm_dir}")
    if VQC_TEST_LIMIT is not None:
        files = files[:VQC_TEST_LIMIT]
    print(f"  [VQC] loading {len(files)} QASM file(s) from {qasm_dir}")

    pubs: List[Tuple[QuantumCircuit]] = []
    meta: List[Dict[str, Any]] = []
    for path in files:
        idx = _parse_vqc_index(path)
        label = _parse_vqc_label(path)
        qc = load_qasm3_circuit(path)
        pubs.append((pm.run(qc),))
        meta.append(
            {
                "index": idx,
                "true_label": int(label),
                "qasm_file": path.name,
            }
        )

    t0 = time.time()
    pub_results = _sampler_chunks(pubs, sampler)
    dt = time.time() - t0

    counts_dir = out_vqc / "counts"
    counts_dir.mkdir(exist_ok=True)
    rows = []
    for m, pr in zip(meta, pub_results):
        bits = _bitstrings_from_pub_result(pr)
        cdict: Dict[str, int] = {}
        for b in bits:
            cdict[b] = cdict.get(b, 0) + 1
        idx = m["index"]
        (counts_dir / f"qvc_test_{idx:03d}.json").write_text(json.dumps(cdict))
        ev = parity_expectation_from_bitstrings(bits)
        pred = 1 if ev >= 0 else -1
        rows.append({**m, "ev": ev, "pred": pred})

    y_true = np.array([r["true_label"] for r in rows], dtype=int)
    y_pred = np.array([r["pred"] for r in rows], dtype=int)
    acc = float(np.mean(y_true == y_pred))
    mean_abs = float(np.mean([abs(r["ev"]) for r in rows]))

    summary = {
        "benchmark": "vqc_inference_from_qasm",
        "backend": backend.name,
        "backend_name_arg": backend_name,
        "qasm_dir": str(qasm_dir),
        "shots": SHOTS_VQC,
        "n_images": len(rows),
        "accuracy": acc,
        "mean_abs_parity_ev": mean_abs,
        "wall_time_s": dt,
        "utc_finished": datetime.now(timezone.utc).isoformat(),
    }
    (out_vqc / "summary.json").write_text(json.dumps(summary, indent=2))

    with (out_vqc / "predictions.csv").open("w", newline="") as f:
        w = csv.DictWriter(
            f,
            fieldnames=["index", "true_label", "qasm_file", "ev", "pred"],
        )
        w.writeheader()
        for r in rows:
            w.writerow(r)

    with (out_vqc / "manifest.csv").open("w", newline="") as f:
        w = csv.DictWriter(
            f, fieldnames=["file", "index", "true_label", "qasm_file"],
        )
        w.writeheader()
        for r in rows:
            w.writerow(
                {
                    "file": f"qvc_test_{r['index']:03d}.json",
                    "index": r["index"],
                    "true_label": r["true_label"],
                    "qasm_file": r["qasm_file"],
                }
            )

    print(f"  [VQC] {backend.name}: accuracy={acc*100:.2f}%  "
          f"mean|⟨Z^10⟩|={mean_abs:.4f}  time={dt:.1f}s  -> {out_vqc}")
    return summary


# ---------------------------------------------------------------------------
# XEB
# ---------------------------------------------------------------------------
def run_xeb_braket_from_qasm(
    backend_name: str,
    out_xeb: Path,
    qasm_dir: Path,
) -> Dict[str, Any]:
    out_xeb.mkdir(parents=True, exist_ok=True)
    backend = _get_braket_backend(backend_name)
    pm = generate_preset_pass_manager(target=backend.target,
                                      optimization_level=TRANSPILE_OPTLEVEL)
    sampler = BackendSamplerV2(backend=backend,
                               options={"default_shots": SHOTS_XEB})

    files = sorted(qasm_dir.glob("xeb_d*_c*.qasm"),
                   key=lambda p: _parse_xeb_depth_idx(p))
    if not files:
        raise FileNotFoundError(f"No xeb_d*_c*.qasm files under {qasm_dir}")

    # Group by depth and (optionally) trim per-depth count.
    groups: Dict[int, List[Path]] = {}
    for p in files:
        d, _ = _parse_xeb_depth_idx(p)
        groups.setdefault(d, []).append(p)
    if XEB_PER_DEPTH_LIMIT is not None:
        groups = {d: ps[:XEB_PER_DEPTH_LIMIT] for d, ps in groups.items()}
    depths = sorted(groups.keys())
    counts_per_depth = {d: len(ps) for d, ps in groups.items()}
    print(f"  [XEB] depths={depths}  circuits/depth={counts_per_depth}")

    per_depth: Dict[int, Dict[str, Any]] = {}
    t0 = time.time()

    for d in depths:
        pubs: List[Tuple[QuantumCircuit]] = []
        p_ideals: List[np.ndarray] = []
        circ_files: List[str] = []
        for path in groups[d]:
            qc_meas = load_qasm3_circuit(path)
            qc_no_meas = strip_final_measurements(qc_meas)
            p_ideals.append(ideal_probabilities(qc_no_meas))
            pubs.append((pm.run(qc_meas),))
            circ_files.append(path.name)

        pub_results = _sampler_chunks(pubs, sampler)
        fvals: List[float] = []
        for k, pr in enumerate(pub_results):
            bits = _bitstrings_from_pub_result(pr)
            samples = np.fromiter((int(b, 2) for b in bits),
                                  dtype=np.int64, count=len(bits))
            fvals.append(xeb_fidelity(p_ideals[k], samples))
        per_depth[int(d)] = {
            "mean": float(np.mean(fvals)),
            "sem": float(np.std(fvals) / np.sqrt(len(fvals))) if len(fvals) > 1 else 0.0,
            "per_circuit": fvals,
            "files": circ_files,
        }

    dt = time.time() - t0
    depths_arr = np.array(depths, dtype=float)
    mean_f = np.array([per_depth[d]["mean"] for d in depths])
    sem_f = np.array([per_depth[d]["sem"] for d in depths])

    def model(dd, A, f):
        return A * f ** dd

    fit: Dict[str, Any] = {}
    if len(depths) >= 2:
        try:
            popt, pcov = curve_fit(
                model, depths_arr, mean_f, p0=[1.0, 0.99],
                sigma=np.maximum(sem_f, 1e-3), absolute_sigma=False,
                bounds=([0, 0], [1.5, 1.0]),
            )
            A_fit, f_fit = popt
            A_err, f_err = np.sqrt(np.diag(pcov))
            fit = {"A": A_fit, "A_err": A_err,
                   "f_per_cycle": f_fit, "f_err": f_err}
        except Exception as e:
            fit = {"error": str(e)}
    else:
        fit = {"error": "need >=2 distinct depths to fit f^d"}

    summary = {
        "benchmark": "xeb_from_qasm",
        "backend": backend.name,
        "backend_name_arg": backend_name,
        "qasm_dir": str(qasm_dir),
        "shots": SHOTS_XEB,
        "depths": depths,
        "circuits_per_depth": counts_per_depth,
        "per_depth_mean_sem": {
            str(k): {"mean": per_depth[k]["mean"], "sem": per_depth[k]["sem"]}
            for k in per_depth
        },
        "fit_A_f": fit,
        "wall_time_s": dt,
        "utc_finished": datetime.now(timezone.utc).isoformat(),
    }
    (out_xeb / "summary.json").write_text(json.dumps(summary, indent=2))
    light = {
        str(k): {kk: vv for kk, vv in v.items()
                 if kk not in ("per_circuit", "files")}
        for k, v in per_depth.items()
    }
    (out_xeb / "per_depth_mean_sem.json").write_text(json.dumps(light, indent=2))
    (out_xeb / "per_depth_per_circuit.json").write_text(
        json.dumps(
            {str(k): {"fidelities": v["per_circuit"], "files": v["files"]}
             for k, v in per_depth.items()},
            indent=2,
        )
    )

    plt.figure(figsize=(7, 4))
    plt.errorbar(depths, mean_f, yerr=sem_f, fmt="s-", label=r"$F_{\mathrm{XEB}}$")
    if "f_per_cycle" in fit:
        dd = np.linspace(min(depths), max(depths), 100)
        plt.plot(dd, model(dd, fit["A"], fit["f_per_cycle"]), "k:",
                 label=f"fit f={fit['f_per_cycle']:.4f}")
    plt.xlabel("cycles d")
    plt.ylabel(r"$F_{\mathrm{XEB}}$")
    plt.title(f"XEB — {backend.name}  (from IBM QASM)")
    plt.legend()
    plt.grid(alpha=0.3)
    plt.tight_layout()
    plt.savefig(out_xeb / "xeb_plot.png", dpi=150)
    plt.close()

    print(f"  [XEB] {backend.name}: fit={fit}  time={dt:.1f}s  -> {out_xeb}")
    return summary


# ---------------------------------------------------------------------------
# Per-device runner
# ---------------------------------------------------------------------------
def run_braket_device(backend_name: str) -> Path:
    slug = _device_slug(backend_name)
    out = OUTPUT_ROOT / slug
    (out / "vqc").mkdir(parents=True, exist_ok=True)
    (out / "xeb").mkdir(parents=True, exist_ok=True)

    qasm_dir = _resolve_qasm_dir()
    print(f"\n=== Device: {backend_name}  (slug={slug}) ===")
    print(f"     QASM source: {qasm_dir}")

    run_vqc_braket_from_qasm(backend_name, out / "vqc", qasm_dir)
    run_xeb_braket_from_qasm(backend_name, out / "xeb", qasm_dir)

    (out / "device_index.json").write_text(
        json.dumps(
            {
                "backend_name_arg": backend_name,
                "slug": slug,
                "qasm_dir": str(qasm_dir),
                "vqc_dir": str((out / "vqc").resolve()),
                "xeb_dir": str((out / "xeb").resolve()),
            },
            indent=2,
        )
    )
    print(f"=== finished {backend_name} -> {out.resolve()} ===\n")
    return out


def run_all_listed_devices() -> None:
    if not BRAKET_DEVICES:
        raise ValueError("BRAKET_DEVICES is empty — add at least one backend name.")
    index: Dict[str, str] = {}
    for name in BRAKET_DEVICES:
        p = run_braket_device(name)
        index[_device_slug(name)] = str(p.resolve())
    (OUTPUT_ROOT / "all_devices_index.json").write_text(json.dumps(index, indent=2))
    print("Wrote master index:", (OUTPUT_ROOT / "all_devices_index.json").resolve())
    if _TRACKER is not None:
        try:
            print("Braket tracker:", _TRACKER.quantum_tasks_statistics())
        except Exception as _e:
            print("Braket tracker (n/a):", _e)


## 1) Sanity-check the QASM source folder

Run this first to verify:

- the notebook can locate `data/circuits/`,
- every `qvc_test_*.qasm` parses and exposes a `true label` header,
- every `xeb_d*_c*.qasm` parses and contains exactly 10 qubits.


In [ ]:
qasm_dir = _resolve_qasm_dir()
print("QASM_DIR =", qasm_dir)

vqc_files = sorted(qasm_dir.glob("qvc_test_*.qasm"),
                   key=lambda p: _parse_vqc_index(p))
xeb_files = sorted(qasm_dir.glob("xeb_d*_c*.qasm"),
                   key=lambda p: _parse_xeb_depth_idx(p))

print(f"Found {len(vqc_files)} qvc_test_*.qasm")
for p in vqc_files[:5]:
    idx = _parse_vqc_index(p)
    lab = _parse_vqc_label(p)
    print(f"   {p.name}: index={idx} true_label={lab}")
if len(vqc_files) > 5:
    print(f"   ... ({len(vqc_files) - 5} more)")

print(f"Found {len(xeb_files)} xeb_d*_c*.qasm")
depths_seen = sorted({_parse_xeb_depth_idx(p)[0] for p in xeb_files})
print("   distinct depths:", depths_seen)
for p in xeb_files[:5]:
    d, k = _parse_xeb_depth_idx(p)
    print(f"   {p.name}: depth={d} circuit_index={k}")
if len(xeb_files) > 5:
    print(f"   ... ({len(xeb_files) - 5} more)")

# Spot-check: load one of each and confirm shape.
if vqc_files:
    qc = load_qasm3_circuit(vqc_files[0])
    print(f"Sample VQC circuit ({vqc_files[0].name}): "
          f"{qc.num_qubits}q, {qc.num_clbits}c, depth={qc.depth()}, "
          f"size={qc.size()}")
if xeb_files:
    qc = load_qasm3_circuit(xeb_files[0])
    qc_nm = strip_final_measurements(qc)
    p_ideal = ideal_probabilities(qc_nm)
    print(f"Sample XEB circuit ({xeb_files[0].name}): "
          f"{qc.num_qubits}q, depth={qc.depth()}, size={qc.size()}; "
          f"|p_ideal| sum = {p_ideal.sum():.4f} (should be ~1.0)")


## 2) Run **all** backends in `BRAKET_DEVICES` (one shot)

Run **section 1** first to make sure the QASM source is found and parses cleanly.


In [ ]:
run_all_listed_devices()


## 3) Optional — run **one** backend in isolation

Uncomment and change the name, then run only this cell (skip the previous cell
if you do not want to re-run the whole list).


In [ ]:
# CURRENT_DEVICE = "SV1"
# run_braket_device(CURRENT_DEVICE)


## Output layout (for analysis in pandas)

Everything lives under `braket_runs_from_qasm/<UTC_timestamp>/`:

| Path | Contents |
|------|----------|
| `all_devices_index.json` | slug → absolute path (after running the batch cell) |
| `<device_slug>/device_index.json` | pointers to `vqc/` and `xeb/`, plus the `qasm_dir` that was used |
| `<device_slug>/vqc/summary.json` | accuracy, mean `\|⟨Z^10⟩\|`, wall time, shots, source `qasm_dir` |
| `<device_slug>/vqc/predictions.csv` | per-image parity EV + predicted label + source QASM filename |
| `<device_slug>/vqc/counts/qvc_test_NNN.json` | histogram counts (same schema as IBM pipeline / sibling notebook) |
| `<device_slug>/xeb/summary.json` | per-depth mean/SEM + exponential fit |
| `<device_slug>/xeb/per_depth_mean_sem.json` | compact table without per-circuit arrays |
| `<device_slug>/xeb/per_depth_per_circuit.json` | raw fidelity list per depth + source QASM filenames |
| `<device_slug>/xeb/xeb_plot.png` | quick figure |

This layout deliberately mirrors `braket-benchmarks.ipynb` so the same analysis
notebooks (`analysis/vqc_braket_*_analysis.ipynb`) can consume both
output trees. The only structural addition is the `qasm_file` column in
`predictions.csv` / `manifest.csv` and the `files` list under
`per_depth_per_circuit.json`, which tie each row back to the exact `.qasm`
source on disk.
